# 10 · Navier–Stokes with PINNs — Reproducing the Raissi et al. (2019) Methodology

**Level:** Research
**Prerequisites:** entire Beginner–Advanced track, especially `07_inverse_parameter_estimation`

The headline result of Raissi, Perdikaris & Karniadakis (2019) is inferring the
**pressure field and viscosity** of an incompressible flow from **sparse velocity
measurements only** — pressure is never measured, yet the PINN reconstructs it
from the momentum equations. Their demonstration used a cylinder-wake dataset.

Here we reproduce their exact **methodology** on a problem with a known analytical
solution so the notebook is fully self-contained and runnable: the **2-D
Taylor–Green vortex**, an exact decaying solution of the incompressible
Navier–Stokes equations.

### Faithful ingredients from the paper
1. **Stream-function formulation** `u = ψ_y, v = -ψ_x` → incompressibility holds by construction.
2. Network outputs `(ψ, p)`; velocities obtained by autograd.
3. Momentum-equation residuals as the physics loss.
4. Viscosity `ν` treated as an **unknown learnable parameter** (inverse problem).
5. Training data = sparse `(u, v)` samples; pressure is *inferred*, recovered up to a constant.

### References
- Raissi, Perdikaris & Karniadakis (2019), *J. Comput. Phys.* 378:686-707
- Raissi, Yazdani & Karniadakis (2020), *Hidden fluid mechanics*, Science 367:1026-1030
- Cai et al. (2021/2022), *PINNs for fluid mechanics: A review*, Acta Mech. Sinica


## 1 · The Taylor–Green vortex (exact NS solution)

On $[0,2\pi]^2$ with kinematic viscosity $\nu$:

$$u=-\cos x\,\sin y\,e^{-2\nu t},\quad
v=\ \sin x\,\cos y\,e^{-2\nu t},\quad
p=-\tfrac14(\cos 2x+\cos 2y)\,e^{-4\nu t}.$$

These satisfy incompressible Navier–Stokes:

$$u_t+u u_x+v u_y=-p_x+\nu(u_{xx}+u_{yy}),\quad
v_t+u v_x+v v_y=-p_y+\nu(v_{xx}+v_{yy}),\quad u_x+v_y=0.$$

We take the true viscosity $\nu^\star=0.05$, sample sparse noisy $(u,v)$, and ask
the PINN to (a) reconstruct $u,v,p$ everywhere and (b) recover $\nu$.


In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(0); np.random.seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

nu_true = 0.05
def tg(x, y, t):
    e2 = np.exp(-2*nu_true*t); e4 = np.exp(-4*nu_true*t)
    u = -np.cos(x)*np.sin(y)*e2
    v =  np.sin(x)*np.cos(y)*e2
    p = -0.25*(np.cos(2*x)+np.cos(2*y))*e4
    return u, v, p


In [ ]:
# --- Sparse, noisy velocity training data ---
N_data = 2000
xd = np.random.rand(N_data)*2*np.pi
yd = np.random.rand(N_data)*2*np.pi
td = np.random.rand(N_data)*1.0
ud, vd, _ = tg(xd, yd, td)
ud += 0.01*np.random.randn(N_data); vd += 0.01*np.random.randn(N_data)

to_t = lambda a: torch.tensor(a, dtype=torch.float32, device=device).view(-1,1)
xd_t, yd_t, td_t = to_t(xd), to_t(yd), to_t(td)
xd_t.requires_grad_(True); yd_t.requires_grad_(True)   # needed: u=ψ_y, v=-ψ_x at data points
ud_t, vd_t = to_t(ud), to_t(vd)

# collocation points for the physics residual
N_f = 6000
xf = torch.rand(N_f,1,device=device)*2*np.pi
yf = torch.rand(N_f,1,device=device)*2*np.pi
tf = torch.rand(N_f,1,device=device)*1.0
for z in (xf,yf,tf): z.requires_grad_(True)


In [ ]:
class NSNet(nn.Module):
    # inputs (x,y,t) -> outputs (psi, p)
    def __init__(self, layers):
        super().__init__()
        seq=[]
        for i in range(len(layers)-1):
            seq.append(nn.Linear(layers[i],layers[i+1]))
            if i<len(layers)-2: seq.append(nn.Tanh())
        self.net=nn.Sequential(*seq)
        for mo in self.net:
            if isinstance(mo,nn.Linear):
                nn.init.xavier_normal_(mo.weight); nn.init.zeros_(mo.bias)
        self.log_nu = nn.Parameter(torch.tensor(np.log(0.2), dtype=torch.float32))  # wrong start
    @property
    def nu(self): return torch.exp(self.log_nu)
    def forward(self,x,y,t): return self.net(torch.cat([x,y,t],1))

model = NSNet([3,60,60,60,60,2]).to(device)

def grad(o,i): return torch.autograd.grad(o,i,torch.ones_like(o),create_graph=True)[0]

def uvp(model,x,y,t):
    out = model(x,y,t); psi, p = out[:,0:1], out[:,1:2]
    u =  grad(psi,y)               # u = ψ_y
    v = -grad(psi,x)               # v = -ψ_x  -> divergence-free by construction
    return u, v, p


In [ ]:
def physics_residual(model,x,y,t):
    out = model(x,y,t); psi, p = out[:,0:1], out[:,1:2]
    u =  grad(psi,y); v = -grad(psi,x)
    u_t=grad(u,t); u_x=grad(u,x); u_y=grad(u,y)
    v_t=grad(v,t); v_x=grad(v,x); v_y=grad(v,y)
    u_xx=grad(u_x,x); u_yy=grad(u_y,y)
    v_xx=grad(v_x,x); v_yy=grad(v_y,y)
    p_x=grad(p,x); p_y=grad(p,y)
    nu = model.nu
    f_u = u_t + u*u_x + v*u_y + p_x - nu*(u_xx+u_yy)
    f_v = v_t + u*v_x + v*v_y + p_y - nu*(v_xx+v_yy)
    return f_u, f_v

def loss_fn():
    u_p, v_p, _ = uvp(model, xd_t, yd_t, td_t)
    L_data = torch.mean((u_p-ud_t)**2) + torch.mean((v_p-vd_t)**2)
    f_u, f_v = physics_residual(model, xf, yf, tf)
    L_phys = torch.mean(f_u**2) + torch.mean(f_v**2)
    return L_data + L_phys, L_data.item(), L_phys.item()


> **Runtime note.** Full training is ~20k Adam iterations (a few minutes on GPU,
> longer on CPU). The cell below prints the recovered viscosity periodically. On a
> CPU you can lower `EPOCHS` to see the trend; the estimate keeps improving with
> more iterations and an optional L-BFGS finish.

In [ ]:
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
EPOCHS = 20000
nu_hist=[]
for ep in range(EPOCHS):
    opt.zero_grad(); loss, ld, lp = loss_fn(); loss.backward(); opt.step()
    nu_hist.append(model.nu.item())
    if ep % 2500 == 0:
        print(f"epoch {ep:5d} | loss {loss.item():.3e} | data {ld:.2e} phys {lp:.2e} | nu {model.nu.item():.4f}")
print(f"\nRecovered nu = {model.nu.item():.4f} (true {nu_true})")


In [ ]:
# --- Reconstruct fields at t=0 and compare (pressure up to a constant) ---
n=60; gx=np.linspace(0,2*np.pi,n); gy=np.linspace(0,2*np.pi,n)
X,Y=np.meshgrid(gx,gy); Tt=np.zeros_like(X)
xt,yt,tt=to_t(X.ravel()),to_t(Y.ravel()),to_t(Tt.ravel())
xt.requires_grad_(True); yt.requires_grad_(True); tt.requires_grad_(True)
u_pr,v_pr,p_pr = uvp(model,xt,yt,tt)
u_pr=u_pr.detach().cpu().numpy().reshape(n,n)
p_pr=p_pr.detach().cpu().numpy().reshape(n,n)
ue,ve,pe = tg(X,Y,Tt)
p_pr -= p_pr.mean(); pe -= pe.mean()      # pressure is only defined up to a constant

fig,ax=plt.subplots(2,2,figsize=(11,9))
for a,(d,ttl) in zip(ax.ravel(),
        [(ue,'u exact'),(u_pr,'u PINN'),(pe,'p exact (mean-removed)'),(p_pr,'p PINN (inferred!)')]):
    im=a.pcolormesh(X,Y,d,shading='auto',cmap='RdBu_r'); a.set_title(ttl); fig.colorbar(im,ax=a)
plt.tight_layout(); plt.show()
print("pressure rel L2:", np.linalg.norm(p_pr-pe)/np.linalg.norm(pe))


In [ ]:
plt.figure(figsize=(6,4))
plt.plot(nu_hist, lw=2); plt.axhline(nu_true, color='k', ls='--', label='true nu')
plt.xlabel('epoch'); plt.ylabel('estimated nu'); plt.legend()
plt.title('Viscosity recovered from velocity data alone'); plt.tight_layout(); plt.show()


## 2 · What makes this a faithful reproduction

- **Pressure is never in the data.** It is reconstructed purely because it is the
  only field consistent with the momentum equations and the observed velocities —
  exactly the "hidden" quantity Raissi et al. recover.
- **Stream-function trick** enforces `∇·u = 0` exactly, removing the continuity
  residual and improving conditioning.
- **Inverse viscosity** shows the same machinery discovers a physical constant.

To reproduce the *original* cylinder-wake result, swap the Taylor–Green data
generator for the authors' `cylinder_nektar_wake.mat` dataset (linked in the
bibliography) and keep everything else. The reported viscosity/`Re` recovery is
within ~1% of truth.

## 3 · Extensions
1. Add L-BFGS after Adam and measure the accuracy jump.
2. Use the real cylinder-wake dataset and reproduce Table 1 of the paper.
3. Replace the plain MLP with a modified-MLP / Fourier-feature backbone
   (notebook 12) and compare convergence.
